# Shopper Spectrum

## Product Recommendation

In the previous notebook, we segmented customers based on their purchasing behaviour using the K-Means clustering algorithm.

In this notebook, we will build a **Product Recommendation System** using **Item-Based Collaborative Filtering**. The recommendation model identifies products that are frequently purchased together by comparing customer purchase patterns using **Cosine Similarity**.

## Project Objectives

The objectives of this notebook are to:

- Load the cleaned transaction dataset prepared in the previous notebook.
- Prepare customer purchase data for recommendation analysis.
- Create a Customer-Product interaction matrix.
- Measure customer similarity using Cosine Similarity.
- Build a Customer-Based Collaborative Filtering recommendation system.
- Generate personalized product recommendations for customers.
- Demonstrate how recommendation systems support business decision-making.

In [3]:
# Import libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity

import joblib

In [4]:
# Load cleaned dataset

df = pd.read_csv("../data/OnlineRetail_cleaned.csv")

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2022-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2022-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2022-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2022-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2022-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [5]:
df.shape

(392692, 9)

In [6]:
# Plot style

plt.style.use("seaborn-v0_8")

sns.set_palette("Set2")

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

## Create the Customer–Product Matrix

To identify similar products, we first create a matrix where:

- Each row represents a customer.
- Each column represents a product.
- The values indicate the quantity purchased.

This matrix forms the foundation of the recommendation system.

In [7]:
# Create customer-product matrix

customer_product_matrix = df.pivot_table(
    index="CustomerID",
    columns="Description",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

customer_product_matrix.head()

Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
CustomerID,,,,,,,,,,,,,,,,,,,,,
12346,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12347,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12348,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12349,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12350,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Calculate Product Similarity

Now that we have created the customer-product matrix, we can measure how similar products are based on customer purchasing behaviour.

We will use **Cosine Similarity**, which compares products by analyzing the purchase patterns of customers.

In [8]:
# Calculate product similarity

product_similarity = cosine_similarity(
    customer_product_matrix.T
)

In [9]:
# Create product similarity DataFrame

similarity_df = pd.DataFrame(
    product_similarity,
    index=customer_product_matrix.columns,
    columns=customer_product_matrix.columns
)

similarity_df.head()

Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
Description,,,,,,,,,,,,,,,,,,,,,
10 COLOUR SPACEBOY PEN,1.000000,0.030829,0.005989,0.001467,0.000000,0.021649,0.016655,0.558159,0.036699,0.013976,...,0.0,0.003133,0.024645,0.302236,0.016603,0.462700,0.003389,0.721928,0.000000,0.000000
12 COLOURED PARTY BALLOONS,0.030829,1.000000,0.049282,0.057428,0.007737,0.006321,0.059429,0.030706,0.020101,0.008785,...,0.0,0.003717,0.016945,0.025296,0.000000,0.027983,0.000000,0.035871,0.000000,0.018714
12 DAISY PEGS IN WOOD BOX,0.005989,0.049282,1.000000,0.490186,0.000000,0.103634,0.056399,0.050523,0.016043,0.008428,...,0.0,0.001326,0.060435,0.078458,0.000000,0.056453,0.000000,0.046778,0.001474,0.162689
12 EGG HOUSE PAINTED WOOD,0.001467,0.057428,0.490186,1.000000,0.000000,0.039582,0.067720,0.068734,0.020805,0.000953,...,0.0,0.006960,0.091329,0.113126,0.000000,0.077150,0.065874,0.071497,0.030950,0.232180
12 HANGING EGGS HAND PAINTED,0.000000,0.007737,0.000000,0.000000,1.000000,0.000000,0.005010,0.000249,0.000916,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001866


## Build the Recommendation Function

Next, we will create a function that recommends products similar to a selected product.

The recommendations will be based on the highest cosine similarity scores.

In [10]:
# Product recommendation function

def recommend_products(product_name, top_n=5):

    similar_products = similarity_df[product_name].sort_values(
        ascending=False
    )

    recommendations = similar_products.iloc[1:top_n + 1]

    return recommendations

## Test the Recommendation System

Let's test the recommendation function using a sample product from the dataset.

In [11]:
# Display a few product names

customer_product_matrix.columns[:10]

Index(['10 COLOUR SPACEBOY PEN', '12 COLOURED PARTY BALLOONS',
       '12 DAISY PEGS IN WOOD BOX', '12 EGG HOUSE PAINTED WOOD',
       '12 HANGING EGGS HAND PAINTED', '12 IVORY ROSE PEG PLACE SETTINGS',
       '12 MESSAGE CARDS WITH ENVELOPES', '12 PENCIL SMALL TUBE WOODLAND',
       '12 PENCILS SMALL TUBE RED RETROSPOT', '12 PENCILS SMALL TUBE SKULL'],
      dtype='str', name='Description')

In [12]:
# Test product recommendations

recommend_products("WHITE HANGING HEART T-LIGHT HOLDER")

Description
GIN + TONIC DIET METAL SIGN         0.750192
RED HANGING HEART T-LIGHT HOLDER    0.658714
WASHROOM METAL SIGN                 0.643520
LAUNDRY 15C METAL SIGN              0.642200
GREEN VINTAGE SPOT BEAKER           0.631463
Name: WHITE HANGING HEART T-LIGHT HOLDER, dtype: float64

## Save the Recommendation Model

Finally, we will save the customer-product matrix and the product similarity matrix.

These files will be used later in the Streamlit dashboard to generate product recommendations without rebuilding the model every time.

In [14]:
# Save customer-product matrix

customer_product_matrix.to_csv(
    "../data/customer_product_matrix.csv"
)

In [15]:
# Save product similarity matrix

joblib.dump(
    similarity_df,
    "../models/product_similarity.pkl"
)

['../models/product_similarity.pkl']

## Final Observation

In this notebook, we built an **Item-Based Collaborative Filtering** recommendation system using **Cosine Similarity**.

The model compares customer purchase patterns to identify products that are frequently purchased together. Given a product, it recommends the most similar products based on historical purchasing behaviour.

The recommendation model and customer-product matrix have been saved for integration into the Streamlit dashboard.